# GDSFactory component to local 3D S-parameters

This notebook prepares a GDSFactory component as ordinary BeamZ geometry and modal ports, inspects the pre-run setup, then runs a local BeamZ S-parameter column. BeamZ executes on the local JAX CPU or GPU; it does not require a cloud account.

Install a released package in a clean environment before running:

```bash
pip install \"beamz[gdsfactory]\"
```


## Goal

The public entry point is `beamz.design.gdsfactory.prepare`. It returns a transparent `PreparedComponent`: its `design`, `ports`, and `simulation_for(...)` values are native BeamZ objects, so mesh, PML, sources, monitors, and runtime remain editable.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    from IPython.display import display
except ImportError:
    display = print

import beamz
from beamz.design.gdsfactory import Settings, prepare

test_mode = os.environ.get("BEAMZ_DOCS_TEST") == "1"
plt.rcParams.update({"figure.dpi": 120})
print(f"Python: {sys.executable}")
print(f"BeamZ: {beamz.__version__} ({Path(beamz.__file__).resolve()})")


## 1. Define the numerical policy

All lengths in `Settings` are SI metres. The normal notebook configuration uses 21 wavelength samples and a conservative 2 ps duration. Documentation test mode deliberately uses one near-center sample and a short run solely to verify the packaged workflow; it is not a convergence setting.


In [ ]:
settings = Settings(
    wavelengths=(1.55e-6, 1.550001e-6) if test_mode else (1.50e-6, 1.60e-6),
    wavelength_points=1 if test_mode else 21,
    xy_padding=1.5e-6 if test_mode else 3.0e-6,
    z_padding=0.5e-6 if test_mode else 1.0e-6,
    pml_thickness=0.5e-6 if test_mode else 1.0e-6,
    run_time=4e-15 if test_mode else 2.0e-12,
)

setup = prepare(
    "straight",
    settings=settings,
    component_settings={"length": 1.0 if test_mode else 10.0},
)

print(setup.component_name)
for port in setup.port_metadata:
    print(
        f"{port.name}: outward={port.outward_direction}, "
        f"inward={port.inward_direction}, axis={port.axis}"
    )

assert {port.name for port in setup.ports} == {"o1", "o2"}
assert {port.axis for port in setup.ports} == {"x"}


## 2. Inspect geometry, source, monitor, and resource plan

The source-free setup is immutable. `simulation_for` adds one native `ModeSource` at the selected port while retaining native `ModeMonitor` devices at every named port. The preview makes PML and placement inspectable before execution.


In [ ]:
simulation = setup.simulation_for("o1")
estimate = setup.estimate_resources("o1")
print(f"Grid cells: {estimate['grid_cells']:,}")
print(f"Planning estimate: {estimate['estimated_memory_gb']:.3f} GB")
print("Source:", simulation.sources[0].signed_direction)
print("Monitors:", [monitor.name for monitor in simulation.monitors])

figure, _ = setup.preview("o1")
display(figure)


## 3. Run and check one S-matrix column

`run_sparameters(["o1"])` runs only the `o1` excitation and returns `S[o1, o1]` and `S[o2, o1]`. The result records the port ordering, source/monitor convention, PML, frequency grid, and BeamZ version in `provenance`. For a full two-port matrix, call `setup.run_sparameters()` with no argument.


In [ ]:
result = setup.run_sparameters(["o1"])
s11 = result.sparameters.s_matrix[("o1", "o1")]
s21 = result.sparameters.s_matrix[("o2", "o1")]

assert np.all(np.isfinite(s11))
assert np.all(np.isfinite(s21))
assert np.all(result.check_passivity()["o1"] <= 1.0 + 1e-6)

print("S11:", s11)
print("S21:", s21)
print("Provenance keys:", sorted(result.provenance))

figure, _ = result.plot_sparameters()
display(figure)


## Next steps

- Replace `straight` with a PDK crossing or coupler, retain explicit `Settings`, and run all port excitations for a full matrix.
- Pass the active PDK `layer_stack` and an explicit `material_map` to `prepare` for stack-aware geometry.
- Sweep `Settings.grid_spec` or `run_time` and compare S-parameters before treating a result as converged.
- Use `result.check_reciprocity()` only after all reciprocal excitation columns have been calculated.
